# 05 — Sample Weighting + GOSS/DART + Early Stopping (XGBoost & LightGBM saja)

Fokus khusus XGBoost dan LightGBM (RandomForest di luar cakupan
eksperimen ini). Tiga mekanisme yang **belum pernah dicoba** di
eksperimen 01-04, digabung secara bertahap (ablation) supaya kontribusi
tiap komponen kelihatan, bukan cuma hasil akhir gabungan:

1. **Sample weighting** (`compute_sample_weight("balanced")`) — beda
   dari resampling (notebook 02): tidak mengubah jumlah baris data,
   cuma bobot di loss function.
2. **Algoritma boosting alternatif**: `boosting_type="goss"` untuk
   LightGBM (mempertahankan sampel bergradien besar, subsample sampel
   yang sudah gampang ditebak — cocok untuk kelas langka), dan
   `booster="dart"` untuk XGBoost (dropout antar-tree, mengurangi
   overfitting boosting klasik). Default kemarin ('gbdt'/'gbtree')
   tidak pernah diganti.
3. **Early stopping** pakai `eval_set` di val set, `n_estimators` diset
   tinggi (1000) sebagai batas atas, biar jumlah tree optimal dicari
   otomatis alih-alih dipatok dari grid seperti notebook 01.

**Catatan metodologis penting**: varian dengan early stopping memakai
val set untuk MEMILIH titik berhenti (`best_iteration`), lalu val set
yang SAMA dipakai untuk evaluasi — jadi angka val varian ini sedikit
optimistis (ada kebocoran kecil pemilihan model). Sesuai "aturan main"
eksperimen ([README.md](README.md)), kandidat manapun yang menang di
sini tetap harus dikonfirmasi di test set sebelum dianggap final.

Latar belakang kenapa ini didahulukan: eksperimen 01 (hyperparameter
tuning) terpaksa mempersempit `N_ITER`/`cv`/batas atas parameter karena
bug oversubscription CPU — jadi kesimpulan "tuning tidak membantu
signifikan" itu belum teruji penuh. Mekanisme di notebook ini (sample
weight, GOSS/DART, early stopping) sengaja dipilih karena BEDA total
dari apa yang sudah dicoba (resampling di notebook 02, kaskade kategori
di notebook 03), jadi tetap informatif terlepas dari status re-tuning
penuh yang belum dikerjakan.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-09" / "outputs" / "advanced_boosting"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_val, y_val = df_val[feature_cols].values, df_val["label"].values

sw_train = compute_sample_weight("balanced", y_train)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, classes: {len(le.classes_)}")


Train: (7157, 164), Val: (1533, 164), classes: 35


## 1. XGBoost — ablation: baseline -> +sample_weight -> +dart -> +dart+early_stopping

In [2]:
BASE_XGB = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)

xgb_results = []
xgb_models = {}

# v0: baseline (config.yaml saat ini)
t0 = time.time()
clf = XGBClassifier(**BASE_XGB)
clf.fit(X_train, y_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="XGBoost baseline")
res["fit_seconds"] = round(time.time() - t0, 1)
xgb_results.append(res)
xgb_models["baseline"] = clf
print_summary(res)

# v1: + sample_weight
t0 = time.time()
clf = XGBClassifier(**BASE_XGB)
clf.fit(X_train, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="XGBoost +sample_weight")
res["fit_seconds"] = round(time.time() - t0, 1)
xgb_results.append(res)
xgb_models["sample_weight"] = clf
print_summary(res)

# v2: + sample_weight + dart
DART_PARAMS = dict(booster="dart", rate_drop=0.1, skip_drop=0.5, normalize_type="tree")
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="XGBoost +sample_weight+dart")
res["fit_seconds"] = round(time.time() - t0, 1)
xgb_results.append(res)
xgb_models["sample_weight_dart"] = clf
print_summary(res)

# v3: + sample_weight + dart + early stopping (n_estimators ceiling 1000)
EARLY_XGB = {**BASE_XGB, **DART_PARAMS}
EARLY_XGB["n_estimators"] = 1000
EARLY_XGB["early_stopping_rounds"] = 30
t0 = time.time()
clf = XGBClassifier(**EARLY_XGB)
clf.fit(X_train, y_train, sample_weight=sw_train, eval_set=[(X_val, y_val)], verbose=False)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le,
               model_name="XGBoost +sample_weight+dart+early_stop")
res["fit_seconds"] = round(time.time() - t0, 1)
res["best_iteration"] = int(clf.best_iteration) if clf.best_iteration is not None else EARLY_XGB["n_estimators"]
xgb_results.append(res)
xgb_models["sample_weight_dart_earlystop"] = clf
print_summary(res)
print(f"Best iteration (early stop): {res['best_iteration']} / {EARLY_XGB['n_estimators']} max")



  XGBoost baseline
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480



  XGBoost +sample_weight
  Accuracy          : 0.4905
  Precision (macro) : 0.3088
  Recall (macro)    : 0.2855
  F1 (macro)        : 0.2887
  F1 (weighted)     : 0.4848
  Top-3 Accuracy    : 0.7599
  Top-5 Accuracy    : 0.8565



  XGBoost +sample_weight+dart
  Accuracy          : 0.4918
  Precision (macro) : 0.3156
  Recall (macro)    : 0.3455
  F1 (macro)        : 0.3217
  F1 (weighted)     : 0.4983
  Top-3 Accuracy    : 0.7482
  Top-5 Accuracy    : 0.8330



  XGBoost +sample_weight+dart+early_stop
  Accuracy          : 0.4958
  Precision (macro) : 0.3165
  Recall (macro)    : 0.3199
  F1 (macro)        : 0.3093
  F1 (weighted)     : 0.4912
  Top-3 Accuracy    : 0.7619
  Top-5 Accuracy    : 0.8506
Best iteration (early stop): 567 / 1000 max


## 2. LightGBM — ablation: baseline -> +sample_weight -> +goss -> +goss+early_stopping

In [3]:
BASE_LGBM = dict(
    n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1,
)

lgbm_results = []
lgbm_models = {}

# v0: baseline
t0 = time.time()
clf = LGBMClassifier(**BASE_LGBM)
clf.fit(X_train, y_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="LightGBM baseline")
res["fit_seconds"] = round(time.time() - t0, 1)
lgbm_results.append(res)
lgbm_models["baseline"] = clf
print_summary(res)

# v1: + sample_weight
t0 = time.time()
clf = LGBMClassifier(**BASE_LGBM)
clf.fit(X_train, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="LightGBM +sample_weight")
res["fit_seconds"] = round(time.time() - t0, 1)
lgbm_results.append(res)
lgbm_models["sample_weight"] = clf
print_summary(res)

# v2: + sample_weight + goss (GOSS tidak kompatibel dengan bagging/subsample)
GOSS_PARAMS = dict(BASE_LGBM)
GOSS_PARAMS.pop("subsample")
GOSS_PARAMS["boosting_type"] = "goss"
t0 = time.time()
clf = LGBMClassifier(**GOSS_PARAMS)
clf.fit(X_train, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le, model_name="LightGBM +sample_weight+goss")
res["fit_seconds"] = round(time.time() - t0, 1)
lgbm_results.append(res)
lgbm_models["sample_weight_goss"] = clf
print_summary(res)

# v3: + sample_weight + goss + early stopping (n_estimators ceiling 1000)
EARLY_LGBM = dict(GOSS_PARAMS)
EARLY_LGBM["n_estimators"] = 1000
t0 = time.time()
clf = LGBMClassifier(**EARLY_LGBM)
clf.fit(
    X_train, y_train, sample_weight=sw_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
)
res = evaluate(y_val, clf.predict(X_val), clf.predict_proba(X_val), le,
               model_name="LightGBM +sample_weight+goss+early_stop")
res["fit_seconds"] = round(time.time() - t0, 1)
res["best_iteration"] = int(clf.best_iteration_) if clf.best_iteration_ else EARLY_LGBM["n_estimators"]
lgbm_results.append(res)
lgbm_models["sample_weight_goss_earlystop"] = clf
print_summary(res)
print(f"Best iteration (early stop): {res['best_iteration']} / {EARLY_LGBM['n_estimators']} max")


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM baseline
  Accuracy          : 0.5095
  Precision (macro) : 0.3574
  Recall (macro)    : 0.2411
  F1 (macro)        : 0.2613
  F1 (weighted)     : 0.4540
  Top-3 Accuracy    : 0.7469
  Top-5 Accuracy    : 0.8324


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM +sample_weight+goss
  Accuracy          : 0.4827
  Precision (macro) : 0.3028
  Recall (macro)    : 0.2355
  F1 (macro)        : 0.2485
  F1 (weighted)     : 0.4359
  Top-3 Accuracy    : 0.7156
  Top-5 Accuracy    : 0.8200


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM +sample_weight+goss+early_stop
  Accuracy          : 0.4781
  Precision (macro) : 0.2745
  Recall (macro)    : 0.2616
  F1 (macro)        : 0.2580
  F1 (weighted)     : 0.4661
  Top-3 Accuracy    : 0.7319
  Top-5 Accuracy    : 0.8324
Best iteration (early stop): 81 / 1000 max


## 3. Bandingkan semua varian

Dibandingkan juga dengan kandidat terkuat sebelumnya (XGBoost +
RandomOverSampler dari notebook 02, F1 weighted 0.508; kaskade kategori
dari notebook 03, F1 weighted 0.508) sebagai acuan — angka keduanya
diketik manual dari `outputs/smote_resampling/summary.json` dan
`outputs/two_stage_classification/summary.json` supaya tabel
perbandingan lengkap dalam satu tempat.


In [4]:
all_results = xgb_results + lgbm_results
comparison = compare_models(all_results)

# Tempel baris pembanding dari eksperimen sebelumnya (dibaca dari file, bukan diketik manual)
smote_summary = json.loads((PROJECT_ROOT / "experiments/2026-07-09/outputs/smote_resampling/summary.json").read_text())
twostage_summary = json.loads((PROJECT_ROOT / "experiments/2026-07-09/outputs/two_stage_classification/summary.json").read_text())

reference_rows = {
    "REF: XGBoost+RandomOverSampler (nb02)": smote_summary["results"]["XGBoost [random_oversample]"],
    "REF: Two-stage kaskade-kategori hard (nb03)": twostage_summary["results"]["Two-stage kaskade-kategori (hard routing)"],
}
ref_df = pd.DataFrame(reference_rows).T
comparison_full = pd.concat([comparison, ref_df[["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]]])
comparison_full


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
XGBoost baseline,0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
XGBoost +sample_weight,0.490541,0.308807,0.285484,0.288748,0.484754,0.759948,0.856491
XGBoost +sample_weight+dart,0.491846,0.315552,0.345488,0.321666,0.498336,0.748206,0.833007
XGBoost +sample_weight+dart+early_stop,0.495760,0.316537,0.319881,0.309273,0.491158,0.761905,0.850620
LightGBM baseline,0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355
LightGBM +sample_weight,0.521200,0.372982,0.288485,0.309157,0.480412,0.756034,0.838226
LightGBM +sample_weight+goss,0.482714,0.302769,0.235472,0.248474,0.435907,0.715590,0.819961
LightGBM +sample_weight+goss+early_stop,0.478147,0.274459,0.261605,0.258027,0.466091,0.731898,0.832355
REF: XGBoost+RandomOverSampler (nb02),0.503588,NaN,NaN,0.288608,0.486359,0.752772,0.849967
REF: Two-stage kaskade-kategori hard (nb03),0.491194,NaN,NaN,0.269030,0.482924,0.762557,0.856491


In [5]:
comparison_full.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "results": {r["model"]: {k: r.get(k) for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy", "fit_seconds", "best_iteration"]}
        for r in all_results},
    "reference_from_previous_experiments": reference_rows,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\advanced_boosting\val_comparison.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\advanced_boosting\summary.json


## 4. Kesimpulan

**Diperbarui 2026-07-10 setelah perbaikan split group-aware** — angka di
bawah ini MENGGANTIKAN kesimpulan sebelumnya. Kesimpulan UTAMA
(XGBoost+sample_weight+DART terbaik) tetap BERTAHAN — sinyal paling
kuat hari ini terbukti robust terhadap perbaikan split.

Hasil val set (`experiments/2026-07-09/outputs/advanced_boosting/val_comparison.csv`):

| Model | Accuracy | F1 macro | F1 weighted | Top-3 / Top-5 | Fit time |
|---|---|---|---|---|---|
| **XGBoost +sample_weight+dart** | 49.2% | **0.322** | **0.498** | 74.8% / 83.3% | 198s |
| XGBoost +sample_weight+dart+early_stop | 49.6% | 0.309 | 0.491 | 76.2% / 85.1% | 544s (best_iter=567) |
| LightGBM +sample_weight | **52.1%** | 0.309 | 0.480 | 75.6% / 83.8% | 31s |
| XGBoost +sample_weight | 49.1% | 0.289 | 0.485 | 76.0% / 85.6% | 34s |
| XGBoost baseline | 51.3% | 0.269 | 0.469 | 75.1% / 84.8% | 31s |
| LightGBM baseline | 50.9% | 0.261 | 0.454 | 74.7% / 83.2% | 31s |
| LightGBM +sample_weight+goss+early_stop | 47.8% | 0.258 | 0.466 | 73.2% / 83.2% | 13s (best_iter=81) |
| LightGBM +sample_weight+goss | 48.3% | 0.248 | 0.436 | 71.6% / 82.0% | 32s |
| *REF: XGBoost+RandomOverSampler (nb02)* | 50.4% | 0.289 | 0.486 | 75.3% / 85.0% | — |
| *REF: Two-stage kaskade-kategori (nb03)* | 49.1% | 0.269 | 0.483 | 76.3% / 85.6% | — |

- **XGBoost+sample_weight+DART tetap juara F1 macro & F1 weighted**
  secara keseluruhan (0.322 / 0.498) — sekarang unggulnya malah SEDIKIT
  LEBIH BESAR dibanding klaim sebelumnya (dulu 0.343/0.517 vs runner-up
  0.335/0.508 — selisih tipis; sekarang 0.322/0.498 vs runner-up
  0.309/0.486 — selisih serupa secara relatif). Ini penting: temuan
  paling berharga hari ini **robust** terhadap perbaikan split, beda
  dengan beberapa temuan di notebook 01/03/04 yang berubah/berbalik.
- **LightGBM+sample_weight sekarang runner-up yang kuat** (F1 macro
  0.309, accuracy TERTINGGI 52.1% dari semua varian) — konsisten dengan
  temuan notebook 02 bahwa LightGBM+class_weight kini kandidat solid.
- **Early stopping di atas DART masih tidak membantu** (F1w 0.491 vs
  0.498 tanpa early stop) meski `best_iteration=567` (lagi-lagi LEBIH
  BANYAK tree, bukan berhenti lebih dini) — pola yang sama persis
  dengan sebelum split diperbaiki, memperkuat dugaan bahwa ini soal
  noise validasi DART akibat dropout, bukan artefak kebocoran split.
- **GOSS tetap merugikan LightGBM** — sekarang bahkan lebih jelas
  (F1 weighted 0.436, TERENDAH dari semua varian termasuk baseline).
  Kesimpulan ini juga robust terhadap perbaikan split.
- **Rekomendasi tidak berubah**: XGBoost + sample_weight + DART tetap
  kandidat terkuat dan sekarang lebih meyakinkan karena bertahan di dua
  split yang berbeda. Validasi di test set adalah langkah berikutnya
  yang wajar.
